# Phase C9 — `r008_liquidity`: 5-Seed Transformer Ensemble with Liquidity-Aware Env

**Setup:** Runtime → GPU (T4 or better).

**What's different from r007:**
- `use_liquidity_features: True` — adds 2 per-asset features (20-day volume z-score + 20-day smoothed OHLC-dispersion spread proxy). Obs width 65 → 75.
- `liquidity_noise_enabled: True` — stochastic fill noise σ_t = 5bps × (1 + 10·spread_proxy_t), seed-reproducible via gym `np_random`.
- All other hyperparameters (seeds, 100k steps, Transformer dims, CVaR λ=0.01, net_arch) identical to r007 so the r007↔r008 ablation is clean (row G in the paper).

**Workflow:**
1. Cell 1: install deps.
2. Cell 2: upload `algo-trading-rl.zip`.
3. Cell 3: verify GPU.
4. Cell 4: patch `configs/transformer.yaml` — flip both Phase-C flags to `True`.
5. Cell 5: train all 5 seeds (~100k steps each, ~5–6 GPU-hrs total on T4).
6. Cell 6: run `ensemble_eval.py` — mean + uncertainty-scaled val/test Sharpe.
7. Cell 7: package `colab_output_r008.zip`. Download and drop into local workspace root.


In [ ]:
!pip install -q gymnasium stable-baselines3 yfinance ta pyyaml


In [ ]:
import os, zipfile, sys
from google.colab import files

print('Select algo-trading-rl.zip when the dialog appears...')
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
for candidate in ('algo-trading-rl', 'algo-trading-rl-main'):
    if os.path.isdir(candidate):
        os.chdir(candidate)
        break
sys.path.insert(0, os.getcwd())
print('Working dir:', os.getcwd())
assert os.path.exists('scripts/train_multi_asset_transformer.py')
assert os.path.exists('scripts/ensemble_eval.py')
assert os.path.exists('envs/multi_asset_env.py')
assert os.path.exists('data/SPY.parquet'), 'data/*.parquet missing from zip'


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('Device:', torch.cuda.get_device_name(0))


In [ ]:
# Flip both Phase-C flags to True in configs/transformer.yaml.
# The trainer reads this file; no CLI override exists, so patching is the
# canonical way to bump from r007 config → r008 config for this run.
import yaml, pprint

CFG_PATH = 'configs/transformer.yaml'
with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['multi_asset']['use_liquidity_features']   = True
cfg['multi_asset']['liquidity_noise_enabled']  = True
# Leave liquidity_noise_base / liquidity_noise_k at env defaults (5e-4, 10.0)
# unless we want to sweep — for r008 we keep defaults so the ablation story is
# 'identical r007 recipe + Phase-C flags on'.

with open(CFG_PATH, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print('multi_asset block after patch:')
pprint.pprint(cfg['multi_asset'])


In [ ]:
# Train 5 seeds. Each writes runs/<ts>_r008_liquidity_seed{N}/ with model_best_val.zip.
# ~100k steps per seed. Serial — Colab has one GPU.
import subprocess

SEEDS = [42, 101, 202, 303, 404]
for s in SEEDS:
    print(f'\n===== SEED {s} =====')
    rc = subprocess.call([
        'python', 'scripts/train_multi_asset_transformer.py',
        '--config', 'configs/transformer.yaml',
        '--tag', f'r008_liquidity_seed{s}',
        '--seed', str(s),
        '--val-eval-freq', '10000',
    ])
    assert rc == 0, f'seed {s} training failed'


In [ ]:
# Collect all r008 runs into a single directory for ensemble_eval.py
import glob, shutil, os
ENS_DIR = 'runs/ensemble_r008_liquidity'
os.makedirs(ENS_DIR, exist_ok=True)
for p in glob.glob('runs/*_r008_liquidity_seed*'):
    dest = os.path.join(ENS_DIR, os.path.basename(p))
    if not os.path.exists(dest):
        shutil.move(p, dest)
print('Members:', sorted(os.listdir(ENS_DIR)))

!python scripts/ensemble_eval.py --run-dir {ENS_DIR} --config configs/transformer.yaml --scale 5.0


In [ ]:
import shutil, os
shutil.make_archive('colab_output_r008', 'zip', 'runs')
print(f'colab_output_r008.zip ({os.path.getsize("colab_output_r008.zip")/1024:.0f} KB)')
from google.colab import files
files.download('colab_output_r008.zip')
